# 01 · Ingestion (DuckDB → Parquet) + leakage-safe splits

Load the raw CSV via DuckDB, persist as Parquet, profile it, then split **before** any transform so no leakage can occur. Splits are stratified to preserve the 0.172% fraud rate.

In [1]:
# --- project-root bootstrap: portable across VS Code / Jupyter / CLI ---
import os
from pathlib import Path
_p = Path.cwd()
while not (_p / 'config' / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)
print('working dir:', Path.cwd())

working dir: /Users/asfalanoi/app_2026/fraud_detection


In [2]:
from fraud.config import load_config
from fraud.data import ingest_csv, stratified_split
from fraud.io import write_parquet, query
from fraud.quality import find_duplicates

cfg = load_config('config/config.yaml')
cfg

Config(seed=42, target='Class', paths=Paths(raw_csv='data/raw/creditcard.csv', parquet='data/interim/creditcard.parquet', processed='data/processed', models='models'), split=Split(train=0.8, valid=0.1, test=0.1), cost=Cost(fp=4.0, k=100))

In [3]:
df = ingest_csv(cfg.paths.raw_csv, cfg.paths.parquet)
print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
query(
    'SELECT COUNT(*) AS rows, SUM(Class) AS frauds, '
    'ROUND(100.0*SUM(Class)/COUNT(*), 4) AS fraud_pct FROM df',
    df=df,
)

,rows,frauds,fraud_pct
0,284807,492.0,0.1727


In [5]:
print('duplicate rows:', find_duplicates(df))

duplicate rows: 1081


In [6]:
train, valid, test = stratified_split(df, target=cfg.target,
    ratios=(cfg.split.train, cfg.split.valid, cfg.split.test), seed=cfg.seed)

for name, part in [('train', train), ('valid', valid), ('test', test)]:
    write_parquet(part, f'{cfg.paths.processed}/{name}.parquet')
    print(f'{name:6s} rows={len(part):>7,d}  fraud_rate={part[cfg.target].mean():.5f}')

train  rows=227,845  fraud_rate=0.00173
valid  rows= 28,481  fraud_rate=0.00172
test   rows= 28,481  fraud_rate=0.00172


In [7]:
import pandas as pd
idx = pd.concat([train, valid, test]).index
assert len(idx) == len(set(idx)) == len(df), 'row overlap detected!'
print('split integrity OK — no overlap, all rows accounted for')

split integrity OK — no overlap, all rows accounted for


**Next:** `02_eda.ipynb` — SQL-driven EDA on the Parquet partitions.